# Figure 3 — Networks: sequence learning and self-orthogonalization

**The network.** $N=64$ conditional units, one per pixel of an $8\times8$ image. Each unit's exposed coordinate
takes $G=21$ equally spaced levels in $[-1,1]$ (the uniform resting law of the construction). The fast state is a
discrete-time jump process: at every renewal all units are redrawn together, each from its local conditional given
the preceding network state $x$,
$$p(\mu_i^+\mid x)\propto \exp\big((h_i+\Pi\,x_i^{\rm obs})\,\mu_i^+-\tfrac{\Pi}{2}\,\mu_i^{+2}\big),\qquad h=Wx,$$
which is the unit's prediction $h_i$ tilted by an observation of value $x_i^{\rm obs}$ with precision $\Pi$. The
state is never reset: when the presentation changes, the network starts from where the previous one left it.

**Learning.** At every renewal, every synapse $W_{ij}$ ($i\neq j$) moves by $\eta\,s_j^+\,(\mu_i^+-L(h_i))$: the
error of unit $i$ against its prediction from the preceding state, paired with the renewed state of unit $j$.
We use the low-noise limit of the structural chain (every synapse takes the mean step of the registered
renewals); the single-unit simulations of Figure 2 show the full stochastic structural chain. Two controls change
only the timing of the two factors: *preceding* pairs the error with $s_j$ instead of $s_j^+$ (the gradient of the
pseudo-log-likelihood), and *reset* sets the fast state to zero at every switch.

**Patterns.** The ten training digits of FEP-ANN simulation 2 (scikit-learn, first example of each digit; pixel
values squared and each image z-scored), observed as $x^{\rm obs}=L_G(2.2\,\xi)$ with precision $\Pi=16$.
$L_G$ is the mean map of the 21-level family, so that the observed value is the mean of the family at a finite
field. Learning rate $\eta=10^{-3}$.

**Readout.** Attractors are the fixed points of $x\leftarrow L_G(Wx)$ reached from $0.3\,x^{\rm obs}/\max|x^{\rm obs}|$.
Because the network has no bias, $-a$ is a fixed point whenever $a$ is; $a$ and $-a$ are one attractor, read with
opposite signs, and are counted once. Orthogonality is the mean deviation from $90^\circ$ over all pairs of
distinct attractors, compared with the same measure for the ten digits.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, jax, jax.numpy as jnp
import matplotlib.pyplot as plt
from network.renewal import learn, free_run, attractors, distinct_attractors, orthogonality, gain_ratio, Lg
from network.digits import load_digits
from network import plotting as P
P.set_style()
DATA = '../data/networks'; os.makedirs(DATA, exist_ok=True)
tr, te = load_digits(); N = tr.shape[1]
PI, LR = 16.0, 1e-3
Xobs = np.asarray(Lg(jnp.array(2.2 * tr)))

## Self-orthogonalization
Random order, ten renewals per presentation, 5000 presentations, six seeds per condition.

In [2]:
fn = f'{DATA}/so.npz'
CONDS = ['lagged', 'preceding', 'reset']
if not os.path.exists(fn):
    Ws, As = {}, {}
    for pi_, c in enumerate(CONDS):
        for s in range(6):
            W = np.asarray(learn(jnp.array(Xobs, jnp.float32), PI, LR, jax.random.PRNGKey(s), jnp.zeros((N, N)), 5000, 10, pi_, False))
            Ws[f'{c}_{s}'] = W; As[f'{c}_{s}'] = attractors(W, [0.3 * x / np.abs(x).max() for x in Xobs])
    np.savez_compressed(fn, **{'W_' + k: v for k, v in Ws.items()}, **{'A_' + k: v for k, v in As.items()})
z = np.load(fn)
so = {c: [] for c in CONDS}
for c in CONDS:
    for s in range(6):
        U, lab = distinct_attractors(z[f'A_{c}_{s}'])
        so[c].append(dict(n=len(U), orth=orthogonality(U), gain=gain_ratio(z[f'W_{c}_{s}'], tr)))
orth_digits = orthogonality(tr)
for c in CONDS:
    print(f"{c:9s} distinct attractors {[r['n'] for r in so[c]]}  orthogonality {[round(r['orth'], 1) for r in so[c]]}  "
          f"gain mean/centred {np.mean([r['gain'] for r in so[c]]):.2f}   (digits {orth_digits:.1f})")
# which digits share an attractor with opposite signs (seed 0 of the network rule)?
U0, lab0 = distinct_attractors(z['A_lagged_0'])
pairs = [(i, j) for i in range(10) for j in range(i + 1, 10) if lab0[i][0] == lab0[j][0] and lab0[i][1] != lab0[j][1]]
print('sign pairs, seed 0:', pairs)

lagged    distinct attractors [9, 8, 8, 9, 9, 9]  orthogonality [20.7, 19.9, 23.6, 20.7, 21.2, 18.5]  gain mean/centred 0.82   (digits 23.3)
preceding distinct attractors [2, 1, 1, 1, 1, 1]  orthogonality [51.7, nan, nan, nan, nan, nan]  gain mean/centred 1.01   (digits 23.3)
reset     distinct attractors [5, 4, 3, 4, 3, 4]  orthogonality [32.8, 41.7, 38.6, 41.4, 38.8, 28.6]  gain mean/centred 0.99   (digits 23.3)
sign pairs, seed 0: [(0, 4)]


## Sequence learning
Digits 1, 2 and 3, three renewals per presentation, 3000 presentations, in cyclic order ($1\to2\to3\to1\dots$) or in
random order. After learning, the network runs without input from the state of digit 1; the overlap of each state
with the three observed patterns tracks what it replays.

In [3]:
fn = f'{DATA}/seq.npz'
idx = [1, 2, 3]; Xs = jnp.array(Xobs[idx], jnp.float32)
if not os.path.exists(fn):
    out = {}
    for cyc in [True, False]:
        W = learn(Xs, PI, LR, jax.random.PRNGKey(0), jnp.zeros((N, N)), 3000, 3, 0, cyc)
        traj = np.asarray(free_run(W, Xs[0], jax.random.PRNGKey(1), 60))
        out[f'W_{cyc}'] = np.asarray(W); out[f'traj_{cyc}'] = traj
    np.savez_compressed(fn, **out)
zs = np.load(fn)
ovl = {c: np.array([[np.corrcoef(t, np.asarray(x))[0, 1] for x in Xs] for t in zs[f'traj_{c}']]) for c in [True, False]}
for c in [True, False]:
    W = zs[f'W_{c}']; win = ovl[c].argmax(1)
    tr_ = [(win[t], win[t + 1]) for t in range(len(win) - 1) if win[t] != win[t + 1]]
    print(f"{'cyclic' if c else 'random'}: antisymmetric/symmetric norm {np.linalg.norm(W - W.T) / np.linalg.norm(W + W.T):.2f}, "
          f"transitions forward {sum((b - a) % 3 == 1 for a, b in tr_)}, backward {sum((b - a) % 3 == 2 for a, b in tr_)}")

cyclic: antisymmetric/symmetric norm 0.69, transitions forward 17, backward 0
random: antisymmetric/symmetric norm 0.23, transitions forward 0, backward 0


## Panels

In [4]:
CM = 'gray_r'
def img(ax, v, vmax=1.0):
    ax.imshow(np.asarray(v).reshape(8, 8), cmap=CM, vmin=-vmax, vmax=vmax, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
DCOL = [P.BLUE, P.ORANGE, P.AQUA]

# A: replay strip (cyclic) -- first 16 renewals
T0 = 16
fig, axes = plt.subplots(1, T0, figsize=(118 * P.MM, 10 * P.MM), gridspec_kw=dict(wspace=0.06))
for t in range(T0):
    img(axes[t], zs['traj_True'][t])
    k = ovl[True][t].argmax()
    for sp in axes[t].spines.values(): sp.set_visible(True); sp.set_color(DCOL[k]); sp.set_linewidth(1.2)
fig.subplots_adjust(left=0.01, right=0.99, top=0.98, bottom=0.02)
P.save_panel(fig, 'fig3_replay')

# A: overlaps over time, cyclic and random
fig, axes = plt.subplots(2, 1, figsize=(136 * P.MM, 28 * P.MM), sharex=True, layout='constrained')
for ax, c, nm in zip(axes, [True, False], ['cyclic order', 'random order']):
    for k in range(3):
        ax.plot(np.arange(1, 61), ovl[c][:, k], color=DCOL[k], lw=1.1, label=f'digit {idx[k]}')
    ax.set_ylim(-1.05, 1.05); ax.set_yticks([-1, 0, 1]); ax.set_ylabel('overlap', fontsize=6)
    ax.text(1.005, 0.5, nm, transform=ax.transAxes, ha='left', va='center', fontsize=6, color=P.INK2)
axes[0].legend(loc='lower left', ncol=3, fontsize=5.5, handlelength=1.2, columnspacing=0.8, bbox_to_anchor=(0, 1.0), borderaxespad=0.1)
axes[1].set_xlabel('renewals without input')
P.save_panel(fig, 'fig3_overlap')

# A: the presented sequence (small)
fig, axes = plt.subplots(1, 3, figsize=(34 * P.MM, 12 * P.MM), gridspec_kw=dict(wspace=0.45))
for k in range(3):
    img(axes[k], Xobs[idx[k]])
    for sp in axes[k].spines.values(): sp.set_visible(True); sp.set_color(DCOL[k]); sp.set_linewidth(1.2)
fig.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02)
P.save_panel(fig, 'fig3_sequence')

# B: digits and their attractors (network rule, seed 0)
A0 = z['A_lagged_0']
fig, axes = plt.subplots(2, 10, figsize=(104 * P.MM, 23 * P.MM), gridspec_kw=dict(wspace=0.08, hspace=0.15))
for k in range(10):
    img(axes[0, k], tr[k], np.abs(tr).max())
    img(axes[1, k], A0[k])
    i_, sg = lab0[k]
    axes[1, k].set_xlabel(f'{"−" if sg < 0 else ""}a{i_ + 1}', fontsize=6, labelpad=1.5, color=P.INK2)
axes[0, 0].set_ylabel('digits', fontsize=6, rotation=0, ha='right', va='center')
axes[1, 0].set_ylabel('attractors', fontsize=6, rotation=0, ha='right', va='center')
fig.subplots_adjust(left=0.12, right=0.995, top=0.99, bottom=0.14)
P.save_panel(fig, 'fig3_attractors')

# C: orthogonality across conditions
fig, ax = P.panel(58, 36)
labels = {'lagged': 'network rule', 'preceding': 'preceding\nboundary', 'reset': 'reset'}
for i, c in enumerate(CONDS):
    v = [r['orth'] for r in so[c] if np.isfinite(r['orth'])]
    ax.scatter(np.full(len(v), i) + np.linspace(-0.15, 0.15, len(v)), v, s=9, color=[P.BLUE, P.MUTED, P.MUTED][i], lw=0, zorder=3)
    nmiss = sum(1 for r in so[c] if not np.isfinite(r['orth']))
    if nmiss: ax.text(i, 3, f'{nmiss}× one\nattractor', ha='center', fontsize=5, color=P.MUTED)
ax.axhline(orth_digits, color=P.INK2, lw=0.8, ls='--'); ax.text(0.55, orth_digits + 1.5, 'digits', fontsize=5.5, color=P.INK2, ha='center')
ax.set_xticks(range(3)); ax.set_xticklabels([labels[c] for c in CONDS], fontsize=5.5)
ax.set_ylabel('|90° − angle| (°)'); ax.set_ylim(0, 75); ax.set_xlim(-0.5, 2.5)
P.save_panel(fig, 'fig3_orthogonality')

'../fig/panels/fig3_orthogonality.svg'

In [5]:
pp = '../fig/panels/'
P.compose('../fig/fig3_networks.svg', 180, 88,
          panels=[(pp + 'fig3_sequence.svg', 2, 7), (pp + 'fig3_replay.svg', 40, 7), (pp + 'fig3_overlap.svg', 40, 17),
                  (pp + 'fig3_attractors.svg', 0, 55), (pp + 'fig3_orthogonality.svg', 116, 49)],
          labels=[('A', 1.5, 4.5), ('B', 1.5, 51.5), ('C', 114, 51.5)],
          texts=[('presented in cyclic order', 4, 22, 6, 'normal'), ('replay without input', 41, 5.5, 6, 'normal')])
json.dump({'so': so, 'orth_digits': orth_digits, 'sign_pairs_seed0': pairs}, open(f'{DATA}/results.json', 'w'), indent=1, default=float)